# TRIBE v2 → SnapUGC engagement on Colab

Runs the [neuro-engagement-forecasting](https://github.com/RehanKrish21/Neuro-engagement-forecasts) pipeline on a Colab GPU:

1. Install TRIBE v2 + this package, log in to Hugging Face
2. Smoke-test TRIBE v2 on one sample video
3. Download SnapUGC, pick a stratified subset, run TRIBE v2 on it (**resumable**: predictions are saved to Google Drive)
4. Compare baseline vs brain vs both for NAWP and ECR

**Runtime:** *Runtime → Change runtime type → GPU*. Pick **A100** or **L4** (Colab Pro). The free **T4** (16 GB) may run out of memory with V-JEPA 2 + Llama 3.2-3B.

**Before you start:** request access to [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) and create a read token at huggingface.co/settings/tokens. Ideally store it as a Colab secret named `HF_TOKEN` (🔑 icon on the left).

TRIBE v2 is CC BY-NC 4.0 (non-commercial). Check SnapUGC's terms too.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install
Takes a few minutes (TRIBE pins torch 2.6, so Colab's torch gets replaced). Full output is shown so any error is visible; if `uv` fails, it retries with plain `pip`.

In [ ]:
PKGS = ('"tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git" '
        '"neuroengage @ git+https://github.com/RehanKrish21/Neuro-engagement-forecasts.git" pyarrow gdown')
!uv pip install --system {PKGS} || pip install {PKGS}

### Restart Python
Run the next cell once the install finished without errors. Colab will say **"Your session crashed"**. That is intentional: it reloads Python so the new torch/numpy versions take effect. Then continue with section 2 (no need to re-run the install).

In [ ]:
import os; os.kill(os.getpid(), 9)

## 2. Drive, paths and Hugging Face login

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/neuroengage")   # persists between sessions
LOCAL = Path("/content/snapugc")                     # fast local disk, wiped on disconnect
CACHE = Path("/content/tribe_cache")                 # TRIBE feature cache (large, local only)
for p in (DRIVE, LOCAL, CACHE):
    p.mkdir(parents=True, exist_ok=True)

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # paste your token when prompted

## 3. Smoke test: one video
First run downloads TRIBE v2 (~1 GB) plus V-JEPA 2, Wav2Vec-BERT and Llama 3.2-3B, so it is slow. Later runs reuse them.

In [ ]:
import time, urllib.request
import pandas as pd
from neuroengage.tribe_runner import TribeExtractor

sample = Path("/content/sample.mp4")
if not sample.exists():
    urllib.request.urlretrieve("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", sample)

extractor = TribeExtractor(cache_folder=CACHE)
t0 = time.time()
smoke = extractor.run(pd.DataFrame({"video_id": ["sintel"], "path": [str(sample)]}), "/content/smoke", overwrite=True)
print(f"{time.time() - t0:.0f} s for a {len(smoke)} s video")
smoke.head()

## 4. Get SnapUGC
Download links come from the [SnapUGC repo](https://github.com/dasongli1/SnapUGC_Engagement/blob/main/dataset/readme.md). The archive is saved to Drive once, then unpacked to local disk each session (much faster than reading videos from Drive).

If `gdown` hits a Google Drive quota error, download the file in your browser, put it in `MyDrive/neuroengage/`, and re-run.

In [ ]:
import gdown, shutil
SPLIT = "train"   # or "val"
FILE_IDS = {"train": "1Mv5Esq5gGuxRTayabRUb5NmHwEN3JdbD", "val": "1iY7qUfqivYzUqUID501dKQDpI3NQil7y"}

dl_dir = DRIVE / f"snapugc_{SPLIT}"
dl_dir.mkdir(exist_ok=True)
existing = [p for p in dl_dir.iterdir() if p.is_file()]
if existing:
    archive = existing[0]
else:  # trailing "/" makes gdown keep the original file name (and extension)
    archive = Path(gdown.download(id=FILE_IDS[SPLIT], output=f"{dl_dir}/", quiet=False))
print("archive:", archive, f"{archive.stat().st_size / 1e9:.1f} GB")

if not any(LOCAL.rglob("*.mp4")):
    shutil.unpack_archive(str(archive), str(LOCAL))  # format inferred from the extension

In [ ]:
import glob, os
from collections import Counter
csvs = sorted(glob.glob(f"{LOCAL}/**/*.csv", recursive=True))
mp4_dirs = Counter(os.path.dirname(p) for p in glob.glob(f"{LOCAL}/**/*.mp4", recursive=True))
print("label files:", csvs)
print("video folders:", mp4_dirs.most_common(3))
LABELS = csvs[0]                       # change if the right file is a different one
VIDEOS = mp4_dirs.most_common(1)[0][0]
pd.read_csv(LABELS).head()

## 5. Pick a subset and run TRIBE v2
Start with `N = 300` to measure speed, then raise it. Re-running this cell **resumes**: finished videos are cached in Drive.

In [ ]:
N = 300
RUN = DRIVE / f"runs/snapugc_{SPLIT}_n{N}"
RUN.mkdir(parents=True, exist_ok=True)
!neuroengage snapugc-prepare --labels "{LABELS}" --videos-dir "{VIDEOS}" --n {N} --out "{RUN}/manifest.csv"

In [ ]:
t0 = time.time()
!neuroengage extract --manifest "{RUN}/manifest.csv" --out "{RUN}/work" --cache "{CACHE}"
done = len(list((RUN / "work" / "preds").glob("*.npz")))
print(f"{done} videos with predictions; this session took {(time.time() - t0) / 60:.1f} min")

## 6. Brain features vs baselines

In [ ]:
!neuroengage train-scores --manifest "{RUN}/manifest.csv" --timeseries "{RUN}/work/network_timeseries" \
    --targets nawp,ecr --baseline-cols duration_s,title_n_words,has_description \
    --title "SnapUGC: TRIBE v2 brain features vs baselines" --out "{RUN}/results"

from IPython.display import Image, Markdown, display
display(Image(str(RUN / "results" / "summary.png")))
display(Markdown((RUN / "results" / "report.md").read_text()))

**Reading the result:** the brain features are useful only if `baseline+brain` beats `baseline` by more than the fold-to-fold spread (`spearman_std`) in most folds. With a few hundred videos, expect wide error bars. Scale `N` up before drawing conclusions. A stronger next baseline is the raw V-JEPA 2 / Wav2Vec-BERT / Llama embeddings TRIBE is built on.